In [1]:
!nvidia-smi

Mon Aug 25 16:01:44 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     On  |   00000000:4F:00.0 Off |                    0 |
|  0%   34C    P8             58W /  300W |       0MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install transformer-lens dictionary-learning



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip


In [3]:
import torch
from transformer_lens import HookedTransformer, HookedTransformerConfig
import numpy as np
import pandas as pd
import ast
from torch.utils.data import Dataset, DataLoader

from huggingface_hub import hf_hub_download


REPO_ID = "sebastianhoenig/2L2H_Final"
FILENAME = "D256_L2_H2_attnOnly1_lr5.0e-04_wd0.01.pt"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
try:
    from IPython.display import clear_output
    clear_output()
except ImportError:
    pass  # clear_output not available, continue anyway

import numpy as np
import pandas as pd
from tqdm import tqdm

E = 100  # num entities
T = 10   # num types/relations

SEP = E + T
Q = E + T + 1
PAD = E + T + 2
D_VOCAB = E + T + 3

IGNORE_INDEX = -100
ENTITIES = np.arange(0, E)
TYPES    = np.arange(E, E + T)

N_WORLDS = 80_000
MIN_FACTS, MAX_FACTS = 4, 8
SEED = 0

rng = np.random.default_rng(SEED)

def produce_example_by_index(idx: int, *, allow_self_loops: bool = False):
    rng = np.random.default_rng(np.random.SeedSequence([BASE_SEED, idx]))

    k = int(rng.integers(MIN_FACTS, MAX_FACTS + 1))

    facts = []
    seen_head_rel = set()
    while len(facts) < k:
        e = int(rng.integers(0, E))
        t = int(rng.integers(0, T)) + E
        if (e, t) in seen_head_rel:
            continue
        e2 = int(rng.integers(0, E))
        while (not allow_self_loops) and e2 == e:
            e2 = int(rng.integers(0, E))
        seen_head_rel.add((e, t))
        facts.append((e, t, e2))

    q_idx = int(rng.integers(0, k))
    Eq, Tq, E2q = facts[q_idx]

    if rng.random() < 0.75 and len(facts) < MAX_FACTS:
        distractor_t = int(rng.integers(0, T)) + E

        while distractor_t == Tq: # Ensure the relation is different
            distractor_t = int(rng.integers(0, T)) + E

        distractor_e2 = int(rng.integers(0, E))
        while distractor_e2 == E2q: # Ensure the tail is different
            distractor_e2 = int(rng.integers(0, E))

        # Add the distractor fact IF it doesn't create a collision
        if (Eq, distractor_t) not in seen_head_rel:
            distractor_fact = (Eq, distractor_t, distractor_e2)

            insert_pos = int(rng.integers(0, len(facts) + 1))
            facts.insert(insert_pos, distractor_fact)

    seq = []
    for (e, t, e2) in facts:
        seq.extend([e, t, e2, SEP])

    seq.extend([Tq, Eq, Q])

    label = E2q
    return seq, label
TOTAL_TRAIN = 16_100_000
BLOCK_SIZE  = 80_000
VAL_SIZE = 20_000
TRAIN_OFFSET = VAL_SIZE
TRAIN_SIZE   = TOTAL_TRAIN
BASE_SEED = 0

class ValDataset(torch.utils.data.Dataset):
    def __len__(self): return VAL_SIZE
    def __getitem__(self, i):
        seq, label = produce_example_by_index(i)
        return torch.tensor(seq, dtype=torch.long), torch.tensor(label, dtype=torch.long)


class TrainStream(torch.utils.data.IterableDataset):
    def __init__(self, block_size=BLOCK_SIZE, offset=TRAIN_OFFSET, size=TRAIN_SIZE):
        super().__init__()
        self.block_size = block_size
        self.offset = offset
        self.size = size
        self._epoch = 0

    def set_epoch(self, epoch:int):
        self._epoch = epoch

    def __iter__(self):
        # compute which block to serve this epoch, with wrap-around
        start_in_train = (self._epoch * self.block_size) % self.size
        # stream exactly block_size samples each epoch
        for i in range(self.block_size):
            local_idx = (start_in_train + i) % self.size
            global_idx = self.offset + local_idx
            seq, label = produce_example_by_index(global_idx)
            x = torch.tensor(seq, dtype=torch.long)
            y = torch.tensor(label, dtype=torch.long)
            yield x, y

    def __len__(self):
        return self.block_size

val_dataset = ValDataset()
train_dataset = TrainStream()

N_LAYERS = 2
HEADS = 2

d_model = 256
n_ctx   = 64

def build_model(n_layers: int, n_heads: int) -> HookedTransformer:
    if d_model % n_heads != 0:
        return None
    d_head = d_model // n_heads

    cfg = HookedTransformerConfig(
        n_layers=n_layers,
        n_heads=n_heads,
        d_model=d_model,
        d_head=d_head,
        n_ctx=n_ctx,
        d_vocab=D_VOCAB,
        d_vocab_out=E,
        attn_only=True,
        normalization_type="LN",
        positional_embedding_type="rotary",
    )
    return HookedTransformer(cfg)

model = build_model(N_LAYERS, HEADS)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
# Load the model
pretrained_weights = torch.load(weights_path, map_location=device, weights_only=True)
state_dict = pretrained_weights["model"]
model.load_state_dict(state_dict)

print("Model loaded successfully.")


def collate_fn(batch):
    max_len = max(len(seq) for seq,_ in batch)
    B = len(batch)
    toks   = torch.full((B, max_len), PAD, dtype=torch.long)
    target = torch.full((B, max_len), IGNORE_INDEX, dtype=torch.long)

    for i, (seq, label) in enumerate(batch):
        x = torch.tensor(seq if isinstance(seq, list) else seq.tolist(), dtype=torch.long)
        L = len(x)
        toks[i, :L] = x
        q_pos = (x == Q).nonzero(as_tuple=False).squeeze()
        assert q_pos.numel() == 1, "Each example must have exactly one Q"
        target[i, q_pos.item()] = int(label)
    return toks, target

Moving model to device:  cuda
Model loaded successfully.


In [4]:
"""### Train a SAE using StandardTrainer with your datasets"""

import torch as t
from dictionary_learning.trainers.standard import StandardTrainer
from dictionary_learning import AutoEncoder, utils
from torch.utils.data import DataLoader

# === Pick activation site ===
layer = 1
act_name = f"blocks.{layer}.hook_resid_post"

# === SAE trainer ===
trainer = StandardTrainer(
    steps=100_000,                  # total training steps
    activation_dim=model.cfg.d_model,
    dict_size=16384,                 # size of dictionary (SAE hidden dim) - reduced for faster training
    layer=layer,
    lm_name="entity_binding_model",  # just a string identifier
    lr=1e-4,
    l1_penalty=1e-3,
    device=device,
    warmup_steps=1000,             # learning rate warmup
    sparsity_warmup_steps=2000,    # sparsity penalty warmup
)

print(f"SAE initialized: dict_size={trainer.ae.dict_size}, activation_dim={trainer.ae.activation_dim}")

# === DataLoaders ===
train_loader = DataLoader(train_dataset, batch_size=64, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=64, collate_fn=collate_fn)

# === Training Loop ===
step = 0
best_loss = float('inf')
best_mse_loss = float('inf')
best_sparsity_loss = float('inf')
epochs_no_improve = 0
early_stopping_patience = 10

print("Starting SAE training...")

for epoch in range(200):   # loop over epochs
    epoch_loss = 0.0
    epoch_mse_loss = 0.0
    epoch_sparsity_loss = 0.0
    num_batches = 0

    # Set epoch for TrainStream
    train_dataset.set_epoch(epoch)

    for toks, _ in train_loader:
        toks = toks.to(device)

        # run model & collect activations
        with torch.no_grad():
            cache = model.run_with_cache(toks, names_filter=[act_name])[1]
            acts = cache[act_name]   # shape: [batch, seq, d_model]
            # Select activations for the last token in each sequence
            acts = acts[:, -1, :]  # shape: [batch, d_model]

        # update SAE
        trainer.update(step, acts)

        if step % 500 == 0:
            log = trainer.loss(acts, step, logging=True)
            current_loss = log.losses['loss']
            current_mse = log.losses['mse_loss']
            current_sparsity = log.losses['sparsity_loss']
            print(f"Step {step}: loss={current_loss:.4f}, mse={current_mse:.4f}, sparsity={current_sparsity:.4f}")

            # Accumulate loss for epoch tracking
            epoch_loss += current_loss
            epoch_mse_loss += current_mse
            epoch_sparsity_loss += current_sparsity
            num_batches += 1

        step += 1
        if step >= trainer.steps:
            break

    if num_batches > 0:
        avg_epoch_loss = epoch_loss / num_batches
        avg_epoch_mse_loss = epoch_mse_loss / num_batches
        avg_epoch_sparsity_loss = epoch_sparsity_loss / num_batches

        print(f"Epoch {epoch}: Avg Loss={avg_epoch_loss:.4f}, Avg MSE={avg_epoch_mse_loss:.4f}, Avg Sparsity={avg_epoch_sparsity_loss:.4f}")

        # Check for improvement
        if avg_epoch_loss < best_loss or avg_epoch_mse_loss < best_mse_loss or avg_epoch_sparsity_loss < best_sparsity_loss:
            best_loss = min(best_loss, avg_epoch_loss)
            best_mse_loss = min(best_mse_loss, avg_epoch_mse_loss)
            best_sparsity_loss = min(best_sparsity_loss, avg_epoch_sparsity_loss)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            print(f"No improvement for {epochs_no_improve} epochs.")

        if epochs_no_improve >= early_stopping_patience:
            print(f"Early stopping triggered after {epoch + 1} epochs.")
            break

    if step >= trainer.steps:
        break


print("SAE training completed!")

SAE initialized: dict_size=16384, activation_dim=256
Starting SAE training...
Step 0: loss=406.8898, mse=406.8898, sparsity=419.0773
Step 500: loss=0.5977, mse=0.5261, sparsity=286.7114
Step 1000: loss=0.2321, mse=0.1069, sparsity=250.4616
Epoch 0: Avg Loss=135.9066, Avg MSE=135.8409, Avg Sparsity=318.7501
Step 1500: loss=0.1682, mse=0.0244, sparsity=191.7170
Step 2000: loss=0.1578, mse=0.0152, sparsity=142.5216
Epoch 1: Avg Loss=0.1630, Avg MSE=0.0198, Avg Sparsity=167.1193
Step 2500: loss=0.1301, mse=0.0124, sparsity=117.7413
Step 3000: loss=0.0979, mse=0.0087, sparsity=89.2157
Step 3500: loss=0.0922, mse=0.0151, sparsity=77.0925
Epoch 2: Avg Loss=0.1067, Avg MSE=0.0120, Avg Sparsity=94.6832
Step 4000: loss=0.0690, mse=0.0097, sparsity=59.3755
Step 4500: loss=0.0570, mse=0.0112, sparsity=45.7661
Epoch 3: Avg Loss=0.0630, Avg MSE=0.0104, Avg Sparsity=52.5708
Step 5000: loss=0.0456, mse=0.0066, sparsity=39.0171
Step 5500: loss=0.0415, mse=0.0071, sparsity=34.4375
Step 6000: loss=0.0352

In [5]:
# === Save trained SAE ===
save_path = f"{act_name}_trained.pt"
t.save(trainer.ae.state_dict(), save_path)
print(f"SAE saved to {save_path}")

SAE saved to blocks.1.hook_resid_post_trained.pt


In [6]:
# === Evaluate on validation set ===
print("\nEvaluating SAE on validation set...")
trainer.ae.eval()
val_losses = []
val_sparsity = []

with torch.no_grad():
    for toks, _ in val_loader:
        toks = toks.to(device)
        cache = model.run_with_cache(toks, names_filter=[act_name])[1]
        acts = cache[act_name]
        # acts = acts.reshape(-1, acts.size(-1))

        # Select activations for the last token in each sequence
        acts = acts[:, -1, :]  # shape: [batch, d_model]

        # Get features and reconstruction
        features = trainer.ae.encode(acts)
        reconstruction = trainer.ae.decode(features)

        # Compute metrics
        mse_loss = torch.nn.functional.mse_loss(acts, reconstruction)
        sparsity = (features == 0).float().mean()

        val_losses.append(mse_loss.item())
        val_sparsity.append(sparsity.item())

avg_val_loss = np.mean(val_losses)
avg_val_sparsity = np.mean(val_sparsity)
print(f"Validation MSE: {avg_val_loss:.6f}")
print(f"Validation Sparsity: {avg_val_sparsity:.4f} ({avg_val_sparsity*100:.2f}%)")


Evaluating SAE on validation set...
Validation MSE: 0.000017
Validation Sparsity: 0.9840 (98.40%)


In [7]:
acts.shape

torch.Size([32, 256])

In [8]:
len(val_dataset)

20000

In [9]:
all_labels = []
for i in range(len(val_dataset)):
    all_labels.append(val_dataset[i][1])
all_labels = torch.stack(all_labels)


In [10]:
all_labels.shape

torch.Size([20000])

In [11]:
# === Extract SAE features from validation set ===
print("\nExtracting SAE features from validation set...")
all_features = []
all_activations = []
all_reconstructions = []
with torch.no_grad():
    for toks, _ in val_loader:
        toks = toks.to(device)
        cache = model.run_with_cache(toks, names_filter=[act_name])[1]
        acts = cache[act_name]
        # acts = acts.reshape(-1, acts.size(-1))

        # Select activations for the last token in each sequence
        acts = acts[:, -1, :]  # shape: [batch, d_model]
        # Extract features
        features = trainer.ae.encode(acts)
        reconstruction = trainer.ae.decode(features)

        all_features.append(features.cpu())
        all_activations.append(acts.cpu())
        all_reconstructions.append(reconstruction.cpu())

# Concatenate all features
all_features = torch.cat(all_features, dim=0)
all_activations = torch.cat(all_activations, dim=0)
all_reconstructions = torch.cat(all_reconstructions, dim=0)

print(f"Extracted features shape: {all_features.shape}")
print(f"Original activations shape: {all_activations.shape}")
print(f"Reconstructions shape: {all_reconstructions.shape}")
print(f"Labels shape: {all_labels.shape}")


Extracting SAE features from validation set...
Extracted features shape: torch.Size([20000, 16384])
Original activations shape: torch.Size([20000, 256])
Reconstructions shape: torch.Size([20000, 256])
Labels shape: torch.Size([20000])


In [12]:
val_dataset[0][0].shape

torch.Size([35])

In [13]:
# === Analyze feature statistics ===
print("\n=== SAE Feature Analysis ===")
sparsity = (all_features == 0).float().mean()
print(f"Overall feature sparsity: {sparsity:.4f} ({sparsity*100:.2f}% of features are zero)")

# Feature activation frequency
feature_activations = (all_features > 0).float()
feature_frequency = feature_activations.mean(dim=0)
print(f"\nTop 10 most active features:")
top_features = feature_frequency.topk(10)
for i, (idx, freq) in enumerate(zip(top_features.indices, top_features.values)):
    print(f"  Feature {idx.item()}: {freq.item():.4f}")

# Sparsity per sample
sparsity_per_sample = (all_features == 0).float().mean(dim=1)
print(f"\nSparsity per sample - Mean: {sparsity_per_sample.mean():.4f}, Std: {sparsity_per_sample.std():.4f}")

# Reconstruction quality
reconstruction_error = torch.norm(all_activations - all_reconstructions, dim=1)
print(f"Reconstruction error - Mean: {reconstruction_error.mean():.4f}, Std: {reconstruction_error.std():.4f}")

print(f"\n✅ SAE training and feature extraction completed!")
print(f"Features saved in 'all_features' tensor with shape {all_features.shape}")
print(f"You can now use these features for interpretability analysis or downstream tasks.")

# Save features to file
features_save_path = f"{act_name}_features.pt"
torch.save({
    'features': all_features,
    'activations': all_activations,
    'labels': all_labels,
    'reconstructions': all_reconstructions,
    'feature_frequency': feature_frequency,
    'sparsity_per_sample': sparsity_per_sample,
    'reconstruction_error': reconstruction_error
}, features_save_path)
print(f"Features saved to {features_save_path}")


=== SAE Feature Analysis ===
Overall feature sparsity: 0.9840 (98.40% of features are zero)

Top 10 most active features:
  Feature 2154: 0.9722
  Feature 980: 0.9484
  Feature 6181: 0.9134
  Feature 1948: 0.8838
  Feature 15372: 0.8455
  Feature 13204: 0.8242
  Feature 10526: 0.7867
  Feature 4278: 0.7850
  Feature 1635: 0.7804
  Feature 1540: 0.7773

Sparsity per sample - Mean: 0.9840, Std: 0.0015
Reconstruction error - Mean: 0.0529, Std: 0.0397

✅ SAE training and feature extraction completed!
Features saved in 'all_features' tensor with shape torch.Size([20000, 16384])
You can now use these features for interpretability analysis or downstream tasks.
Features saved to blocks.1.hook_resid_post_features.pt


In [14]:
from collections import defaultdict, Counter

# === Analyze correlation between features and labels ===
print("\n=== Feature-Label Correlation Analysis ===")

# Find the index of the most activated feature for each sample
most_active_feature_indices = torch.argmax(all_features, dim=1)

# Check the shapes to ensure they match
print(f"Shape of most_active_feature_indices: {most_active_feature_indices.shape}")
print(f"Shape of all_labels: {all_labels.shape}")

labels_to_sae_idx = defaultdict(list)
for feat_idx, label in zip(most_active_feature_indices, all_labels):
  labels_to_sae_idx[label.item()].append(feat_idx.item())
labels_to_sae_idx_cnt = defaultdict(Counter)
for i, occurrences in labels_to_sae_idx.items():
    labels_to_sae_idx_cnt[i] = Counter(occurrences).most_common() # feat_idx, counts


=== Feature-Label Correlation Analysis ===
Shape of most_active_feature_indices: torch.Size([20000])
Shape of all_labels: torch.Size([20000])


In [15]:
labels_to_sae_idx_cnt

defaultdict(collections.Counter,
            {26: [(1540, 135), (2154, 59)],
             25: [(1540, 139), (2154, 72)],
             38: [(1540, 140), (2154, 67)],
             86: [(1540, 126), (2154, 64)],
             65: [(1540, 131), (2154, 65)],
             98: [(1540, 140), (2154, 77)],
             21: [(1540, 125), (2154, 75)],
             53: [(1540, 131), (2154, 67)],
             5: [(1540, 130), (2154, 62)],
             46: [(1540, 146), (2154, 71)],
             80: [(1540, 131), (2154, 63)],
             96: [(1540, 146), (2154, 68)],
             75: [(1540, 120), (2154, 72)],
             58: [(1540, 152), (2154, 68)],
             24: [(1540, 117), (2154, 79)],
             74: [(1540, 128), (8009, 71)],
             57: [(1540, 132), (8128, 71), (2154, 1)],
             64: [(1540, 136), (2154, 71)],
             8: [(1540, 109), (2154, 79)],
             12: [(1540, 118), (2154, 60)],
             31: [(1540, 110), (2154, 87)],
             19: [(1540, 136), (21